## Example 1: Create a certificate authority with certipy

In [ ]:
print("Example 1: Create a certificate authority with certipy")

# The `certipy` on PyPI is LLNL's certificate helper (the tool JupyterHub uses for internal SSL). The
# Active Directory tool that shares the import name (`certipy-ad`, with a `certipy.lib`) is a different project.
import tempfile
import certipy
from certipy import Certipy, CertExistsError
from cryptography import x509

print(f"✓ Certipy package imported: {certipy.__name__}")

with tempfile.TemporaryDirectory() as store_dir:
    cert_store = Certipy(store_dir=store_dir)
    ca_record = cert_store.create_ca('example-ca')
    print(f"✓ CA record: is_ca={ca_record['is_ca']}, files={sorted(ca_record['files'])}")
    assert ca_record['is_ca'] is True and ca_record['parent_ca'] == '', ca_record

    with open(ca_record['files']['cert'], 'rb') as f:
        ca_cert = x509.load_pem_x509_certificate(f.read())
    basic_constraints = ca_cert.extensions.get_extension_for_class(x509.BasicConstraints).value
    print(f"✓ CA certificate: subject={ca_cert.subject.rfc4514_string()}, CA={basic_constraints.ca}")
    assert ca_cert.subject.rfc4514_string() == 'CN=example-ca', ca_cert.subject
    assert ca_cert.issuer == ca_cert.subject, ca_cert.issuer  # self-signed
    assert basic_constraints.ca is True, basic_constraints

    # The store refuses to replace an existing name unless asked to.
    try:
        cert_store.create_ca('example-ca')
    except CertExistsError as e:
        print(f"✓ Duplicate CA rejected as expected: {e}")
    else:
        raise AssertionError("certipy re-created an existing CA without overwrite=True")

print("Example 1 finished")

## Example 2: Sign a server key pair with the certipy CA

In [ ]:
print("Example 2: Sign a server key pair with the certipy CA")

import ipaddress
import tempfile
from certipy import Certipy
from cryptography import x509

with tempfile.TemporaryDirectory() as store_dir:
    cert_store = Certipy(store_dir=store_dir)
    cert_store.create_ca('example-ca')
    server_record = cert_store.create_signed_pair('server', 'example-ca', alt_names=['DNS:localhost', 'IP:127.0.0.1'])
    print(f"✓ Signed pair: parent_ca={server_record['parent_ca']}, files={sorted(server_record['files'])}")
    assert server_record['is_ca'] is False and server_record['parent_ca'] == 'example-ca', server_record

    with open(server_record['files']['cert'], 'rb') as f:
        server_cert = x509.load_pem_x509_certificate(f.read())
    with open(server_record['files']['ca'], 'rb') as f:
        ca_cert = x509.load_pem_x509_certificate(f.read())
    print(f"✓ Server certificate: subject={server_cert.subject.rfc4514_string()}, issuer={server_cert.issuer.rfc4514_string()}")
    assert server_cert.subject.rfc4514_string() == 'CN=server', server_cert.subject
    assert server_cert.issuer == ca_cert.subject, (server_cert.issuer, ca_cert.subject)

    # The CA's key really signed it: this raises if the signature does not verify.
    server_cert.verify_directly_issued_by(ca_cert)
    print("✓ Signature verified against the CA certificate")

    san = server_cert.extensions.get_extension_for_class(x509.SubjectAlternativeName).value
    print(f"✓ Subject alternative names: {san.get_values_for_type(x509.DNSName)} + {san.get_values_for_type(x509.IPAddress)}")
    assert san.get_values_for_type(x509.DNSName) == ['localhost'], san
    assert san.get_values_for_type(x509.IPAddress) == [ipaddress.ip_address('127.0.0.1')], san

print("Example 2 finished")

## Example 3: Reopen the certificate store and look up records

In [ ]:
print("Example 3: Reopen the certificate store and look up records")

import os
import tempfile
from certipy import Certipy, CertNotFoundError

with tempfile.TemporaryDirectory() as store_dir:
    cert_store = Certipy(store_dir=store_dir)
    cert_store.create_ca('example-ca')
    cert_store.create_signed_pair('server', 'example-ca')

    # The store keeps its records in a JSON file inside store_dir, so a new Certipy on the
    # same directory sees everything the first one created.
    reopened = Certipy(store_dir=store_dir)
    names = sorted(reopened.store.store)
    print(f"✓ Records after reopening: {names}")
    assert names == ['example-ca', 'server'], names

    ca_record = reopened.store.get_record('example-ca')
    server_record = reopened.store.get_record('server')
    print(f"✓ example-ca signed: {sorted(ca_record['signees'])}; server's parent CA: {server_record['parent_ca']}")
    assert 'server' in ca_record['signees'], ca_record
    assert server_record['parent_ca'] == 'example-ca', server_record
    assert all(os.path.isfile(path) for path in (server_record['files']['key'], server_record['files']['cert'])), server_record['files']

    # get_files wraps a record's paths in a bundle that knows whether it is a CA.
    assert reopened.store.get_files('example-ca').is_ca() is True
    assert reopened.store.get_files('server').is_ca() is False
    print("✓ get_files: example-ca is a CA, server is not")

    try:
        reopened.store.get_record('no-such-cert')
    except CertNotFoundError as e:
        print(f"✓ Unknown name rejected as expected: {e}")
    else:
        raise AssertionError("certipy returned a record for a name it never created")

print("Example 3 finished")

## Example 4: Build trust bundles

In [ ]:
print("Example 4: Build trust bundles")

import os
import tempfile
from certipy import Certipy
from cryptography import x509

with tempfile.TemporaryDirectory() as store_dir:
    cert_store = Certipy(store_dir=store_dir)
    cert_store.create_ca('root-a')
    cert_store.create_ca('root-b')
    cert_store.create_signed_pair('server', 'root-a')

    # By default create_bundle collects every CA certificate in the store.
    ca_bundle_path = cert_store.create_bundle('ca-bundle.crt')
    with open(ca_bundle_path, 'rb') as f:
        ca_bundle = x509.load_pem_x509_certificates(f.read())
    ca_subjects = sorted(cert.subject.rfc4514_string() for cert in ca_bundle)
    print(f"✓ CA bundle {os.path.basename(ca_bundle_path)}: {ca_subjects}")
    assert ca_subjects == ['CN=root-a', 'CN=root-b'], ca_subjects

    # With explicit names and ca_only=False it bundles exactly the certificates asked for.
    pick_path = cert_store.create_bundle('server-and-root-a.crt', names=['server', 'root-a'], ca_only=False)
    with open(pick_path, 'rb') as f:
        picked = [cert.subject.rfc4514_string() for cert in x509.load_pem_x509_certificates(f.read())]
    print(f"✓ Named bundle: {picked}")
    assert picked == ['CN=server', 'CN=root-a'], picked

    # trust_from_graph creates a CA for each unknown component and writes one trust file per component.
    trust_files = cert_store.trust_from_graph({'web': ['api'], 'api': ['web', 'root-b']})
    trust_file_names = {name: os.path.basename(path) for name, path in sorted(trust_files.items())}
    print(f"✓ Trust files: {trust_file_names}")
    assert sorted(trust_files) == ['api', 'web'], trust_files
    with open(trust_files['api'], 'rb') as f:
        api_trusts = sorted(cert.subject.rfc4514_string() for cert in x509.load_pem_x509_certificates(f.read()))
    assert api_trusts == ['CN=root-b', 'CN=web'], api_trusts
    assert cert_store.store.get_record('web')['is_ca'] is True

print("Example 4 finished")

## Example 5: Remove certificates from the store

In [ ]:
print("Example 5: Remove certificates from the store")

import os
import tempfile
from certipy import Certipy, CertificateAuthorityInUseError, CertNotFoundError

with tempfile.TemporaryDirectory() as store_dir:
    cert_store = Certipy(store_dir=store_dir)
    cert_store.create_ca('example-ca')
    cert_store.create_ca('unused-ca')
    server_record = cert_store.create_signed_pair('server', 'example-ca')
    server_dir = os.path.dirname(server_record['files']['cert'])
    assert os.path.isdir(server_dir), server_dir

    # A CA that has signed certificates is protected from removal.
    try:
        cert_store.store.remove_files('example-ca')
    except CertificateAuthorityInUseError as e:
        print(f"✓ CA in use, removal refused: {e}")
    else:
        raise AssertionError("certipy removed a CA that still has signed certificates")

    # remove_files drops the record and, with delete_dir=True, the directory holding its files.
    removed = cert_store.store.remove_files('server', delete_dir=True)
    print(f"✓ Removed server; its directory still exists: {os.path.isdir(server_dir)}")
    assert removed['files']['cert'] == server_record['files']['cert'] and not os.path.isdir(server_dir), (removed, server_dir)
    try:
        cert_store.store.get_record('server')
    except CertNotFoundError:
        print("✓ server is gone from the store")
    else:
        raise AssertionError("server record survived remove_files")

    # A CA that never signed anything can be removed outright.
    cert_store.store.remove_files('unused-ca', delete_dir=True)
    print(f"✓ Records left: {sorted(cert_store.store.store)}")
    assert sorted(cert_store.store.store) == ['example-ca'], cert_store.store.store

print("Example 5 finished")